In [1]:
import sympy as sp
import numpy as np

ModuleNotFoundError: No module named 'sympy'

In [ ]:
def crear_sistema(ecuaciones, variables):
    variables = sp.symbols(variables)
    ecuaciones_simbolicas = []

    for ecuacion in ecuaciones:
        izquierda, derecha = ecuacion.split("=")

        eq = sp.Eq(
            sp.sympify(izquierda),
            sp.sympify(derecha)
        )

        ecuaciones_simbolicas.append(eq)

    A, b = sp.linear_eq_to_matrix(
        ecuaciones_simbolicas,
        variables
    )

    A = np.array(A).astype(float)
    b = np.array(b).astype(float).flatten()

    return A, b

In [ ]:
def gauss_simple(A, b):
    A = A.astype(float).copy()
    b = b.astype(float).copy()

    n = len(b)

    for k in range(n - 1):

        if abs(A[k, k]) < 1e-12:
            raise ValueError("Pivote cero. No se puede continuar.")

        for i in range(k + 1, n):

            factor = A[i, k] / A[k, k]

            A[i, k:] -= factor * A[k, k:]
            b[i] -= factor * b[k]

    x = np.zeros(n)

    for i in range(n - 1, -1, -1):

        if abs(A[i, i]) < 1e-12:
            raise ValueError("Sistema singular o pivote cero.")

        x[i] = (
            b[i] - np.dot(A[i, i+1:], x[i+1:])
        ) / A[i, i]

    return x


In [ ]:

def gauss_pivote_parcial(A, b):
    A = A.astype(float).copy()
    b = b.astype(float).copy()

    n = len(b)

    for k in range(n - 1):


        fila_pivote = k + np.argmax(
            np.abs(A[k:, k])
        )

        A[[k, fila_pivote]] = A[[fila_pivote, k]]
        b[[k, fila_pivote]] = b[[fila_pivote, k]]

        if abs(A[k, k]) < 1e-12:
            raise ValueError("Sistema singular.")

        for i in range(k + 1, n):

            factor = A[i, k] / A[k, k]

            A[i, k:] -= factor * A[k, k:]
            b[i] -= factor * b[k]

    x = np.zeros(n)

    for i in range(n - 1, -1, -1):

        if abs(A[i, i]) < 1e-12:
            raise ValueError("Sistema singular.")

        x[i] = (
            b[i] - np.dot(A[i, i+1:], x[i+1:])
        ) / A[i, i]

    return x


In [ ]:

def gauss_jordan(A, b):
    A = A.astype(float).copy()
    b = b.astype(float).copy()

    n = len(b)

    for k in range(n):


        fila_pivote = k + np.argmax(
            np.abs(A[k:, k])
        )

        A[[k, fila_pivote]] = A[[fila_pivote, k]]
        b[[k, fila_pivote]] = b[[fila_pivote, k]]

        if abs(A[k, k]) < 1e-12:
            raise ValueError("Sistema singular.")

        factor = A[k, k]

        A[k, :] /= factor
        b[k] /= factor

        for i in range(n):

            if i != k:

                factor = A[i, k]

                A[i, :] -= factor * A[k, :]
                b[i] -= factor * b[k]

    return b


In [ ]:

def gauss_seidel(A, b, x0=None, tol=1e-6, max_iter=100):
    A = A.astype(float).copy()
    b = b.astype(float).copy()

    n = len(b)

    if np.any(np.abs(np.diag(A)) < 1e-12):
        raise ValueError(
            "La diagonal contiene ceros. "
            "Se necesita reordenar el sistema."
        )

    if x0 is None:
        x = np.zeros(n)
    else:
        x = np.array(x0, dtype=float).copy()

    for k in range(max_iter):

        x_anterior = x.copy()

        for i in range(n):

            suma = np.dot(A[i, :], x) - A[i, i] * x[i]

            x[i] = (b[i] - suma) / A[i, i]

        error = np.linalg.norm(x - x_anterior, ord=np.inf)

        if error < tol:
            return x

    print("Se alcanzó el máximo de iteraciones.")

    return x
